# 🧠 Modelo B — Classificador de Parcela de Alto Ticket

**Pergunta de negócio:** dada uma parcela aberta, ela é de **alto ticket** (valor pago acima da mediana da carteira)? Útil para priorizar atendimento e ações de cobrança proativa nas parcelas que mais movem o caixa.

Target binário: `valor_pago > mediana(valor_pago)`. LightGBM. Métrica: ROC-AUC.

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT))

import pandas as pd, numpy as np, joblib
GOLD = PROJECT / 'notebooks' / 'data' / 'gold'
MODELS = PROJECT / 'models'

df = pd.read_parquet(GOLD / 'payments_features.parquet')
df.shape

(100000, 17)

## 1. Construir o target

In [2]:
# Target binário sobre TODAS as parcelas: alto ticket
df['valor_pago'] = df['valor_pago'].fillna(0)
mediana_pagas = df.loc[df['valor_pago'] > 0, 'valor_pago'].median()
df['alto_ticket'] = (df['valor_pago'] > mediana_pagas).astype(int)
df_pagas = df  # mantém variável legado pro fluxo abaixo

print(f'Mediana valor_pago (entre pagas): R$ {mediana_pagas:,.2f}')
print('Distribuição:')
print(df['alto_ticket'].value_counts(normalize=True).round(3))

Mediana valor_pago (entre pagas): R$ 612.79
Distribuição:
alto_ticket
0    0.637
1    0.363
Name: proportion, dtype: float64


## 2. Sinal disponível

In [3]:
df_pagas.select_dtypes('number').corr()['alto_ticket'].sort_values(ascending=False).head(10)

alto_ticket                   1.000000
valor_pago                    0.776364
valor_parcela                 0.552192
dias_atraso_pagamento         0.223210
taxa_adimplencia_historica    0.004571
numero_parcela                0.001809
dias_since_envio              0.000144
score_risco                  -0.000735
Name: alto_ticket, dtype: float64

`valor_parcela` carrega o sinal — parcelas maiores tendem a render pagamentos maiores quando contempladas.

## 3. Features

In [4]:
NUMERIC = ['numero_parcela', 'valor_parcela',
           'taxa_adimplencia_historica', 'dias_since_envio',
           'score_risco']
CATEGORICAL = ['forma_pagamento', 'faixa_valor', 'regiao', 'nome_assessoria']
TARGET = 'alto_ticket'

data = df_pagas.dropna(subset=NUMERIC).copy()
X = data[NUMERIC + CATEGORICAL].copy()
y = data[TARGET]

for c in CATEGORICAL:
    X[c] = X[c].astype('category')

print('Shape:', X.shape, '| pos rate:', y.mean().round(3))

Shape: (100000, 9) | pos rate: 0.363


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

((80000, 9), (20000, 9))

## 4. Treino

In [6]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.04,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=0.3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X_train, y_train, categorical_feature=CATEGORICAL,
          eval_set=[(X_test, y_test)],
          callbacks=[])
print('Treino concluído.')

Treino concluído.


## 5. Avaliação

In [7]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                              classification_report, confusion_matrix, f1_score)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f'Accuracy : {accuracy_score(y_test, pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, proba):.4f}')
print(f'PR-AUC   : {average_precision_score(y_test, proba):.4f}')
print(f'F1-macro : {f1_score(y_test, pred, average="macro"):.4f}')
print()
print(classification_report(y_test, pred, target_names=['Baixo', 'Alto']))

Accuracy : 0.8047
ROC-AUC  : 0.8586
PR-AUC   : 0.7000
F1-macro : 0.7946

              precision    recall  f1-score   support

       Baixo       0.88      0.81      0.84     12739
        Alto       0.70      0.80      0.75      7261

    accuracy                           0.80     20000
   macro avg       0.79      0.80      0.79     20000
weighted avg       0.81      0.80      0.81     20000



In [8]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
imp

dias_since_envio              49230
score_risco                   45665
numero_parcela                38168
taxa_adimplencia_historica    23970
forma_pagamento               10575
valor_parcela                  9594
faixa_valor                    7952
regiao                         1941
nome_assessoria                1905
dtype: int32

## 6. Persistência

In [9]:
artifact = {
    'model': model,
    'numeric_features': NUMERIC,
    'categorical_features': CATEGORICAL,
    'target': 'alto_ticket',
    'target_threshold_brl': float(mediana_pagas),
    'version': '2.0.0',
    'metrics': {
        'accuracy': float(accuracy_score(y_test, pred)),
        'roc_auc':  float(roc_auc_score(y_test, proba)),
        'pr_auc':   float(average_precision_score(y_test, proba)),
        'f1_macro': float(f1_score(y_test, pred, average='macro')),
    },
}
out = MODELS / 'payment_default_v1.joblib'
joblib.dump(artifact, out)
print('Salvo em', out)
print('Métricas:', artifact['metrics'])

Salvo em /home/ivissonalves/Workspaces/previsia/previsia-api/models/payment_default_v1.joblib
Métricas: {'accuracy': 0.8047, 'roc_auc': 0.8586220447281824, 'pr_auc': 0.699964197523637, 'f1_macro': 0.7945566798238755}
